In [6]:
import numpy as np
import pandas as pd
from math import sqrt, exp

In [7]:
def american_binomial_discrete_div(option_type, S, K, T, r, sigma, dividend_dates, dividend_amounts, steps=614):
    dt = T / steps
    u = exp(sigma * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)

    # Convert dividend dates from days to years
    div_times = [x for x in dividend_dates]

    # Prepaid-forward / Escrowed-dividend adjustment
    pv_div_0 = sum(a * exp(-r * t) for t, a in zip(div_times, dividend_amounts) if t <= T)
    S_prepaid = S - pv_div_0

    # PV of remaining dividends at each time step
    pv_remaining = np.zeros(steps + 1)
    for i in range(steps + 1):
        t_now = i * dt
        pv_remaining[i] = sum(
            a * exp(-r * (t_div - t_now))
            for t_div, a in zip(div_times, dividend_amounts)
            if t_div > t_now
        )

    # Stock Values at maturity
    j = np.arange(steps + 1)
    stock = S_prepaid * (u ** j) * (d ** (steps - j)) + pv_remaining[steps]

    if option_type.lower() == "call":
        values = np.maximum(stock - K, 0.0)
    else:
        values = np.maximum(K - stock, 0.0)

    # backward induction with early exercise
    for i in range(steps - 1, -1, -1):
        values = disc * (p * values[1:i + 2] + (1 - p) * values[:i + 1])

        j = np.arange(i + 1)
        stock = S_prepaid * (u ** j) * (d ** (i - j)) + pv_remaining[i]

        if option_type.lower() == "call":
            exercise = np.maximum(stock - K, 0.0)
        else:
            exercise = np.maximum(K - stock, 0.0)

        values = np.maximum(values, exercise)

    return float(values[0])


df = pd.read_csv("/Users/fuyuxuan/Downloads/test12_3.csv")

# Remove blank rows
df = df.dropna(how="all").copy()

results = []

for _, row in df.iterrows():
    option_id = int(row["ID"])
    option_type = row["Option Type"]
    S = float(row["Underlying"])
    K = float(row["Strike"])
    T = float(row["DaysToMaturity"]) / float(row["DayPerYear"])
    day_per_year = float(row["DayPerYear"])
    r = float(row["RiskFreeRate"])
    sigma = float(row["ImpliedVol"])

    dividend_dates = [float(x) / day_per_year for x in str(row["DividendDates"]).split(",")]
    dividend_amounts = [float(x) for x in str(row["DividendAmts"]).split(",")]
    
    value = american_binomial_discrete_div(option_type, S, K, T, r, sigma, dividend_dates, dividend_amounts, steps=614)
    results.append([option_id, value])

out = pd.DataFrame(results, columns=["ID", "Value"])
print(out.to_string(index=False))

 ID     Value
  1 14.503400
  2 11.779086
